In [1]:
%pip install -q --upgrade openai pydantic pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 28.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [2]:
import json
from datetime import datetime, timezone
from typing import List, Literal
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

client = OpenAI(api_key="sk-YOUR_FAKE_API_KEY")
MODEL_NAME = "gpt-4o-mini"


## 1. Schemas Pydantic

In [3]:
class SEOKeyword(BaseModel):
    model_config = ConfigDict(extra="forbid")
    keyword: str = Field(min_length=2, max_length=60)
    search_intent: Literal["informational", "commercial", "transactional"]
    priority: int = Field(ge=1, le=5)

    @field_validator("keyword")
    @classmethod
    def normalize_keyword(cls, value: str) -> str:
        return " ".join(value.lower().strip().split())


class SEOMetadata(BaseModel):
    model_config = ConfigDict(extra="forbid")
    meta_title: str = Field(min_length=30, max_length=60)
    meta_description: str = Field(min_length=120, max_length=160)
    slug: str = Field(pattern=r"^[a-z0-9]+(?:-[a-z0-9]+)*$", max_length=80)
    keywords: List[SEOKeyword] = Field(min_length=5, max_length=8)
    seo_score: int = Field(ge=0, le=100)


class ProductDescription(BaseModel):
    model_config = ConfigDict(extra="forbid")
    product_name: str = Field(min_length=3, max_length=100)
    category: str = Field(min_length=3, max_length=60)
    target_audience: str = Field(min_length=10, max_length=180)
    short_description: str = Field(min_length=80, max_length=220)
    long_description: str = Field(min_length=250, max_length=900)
    key_features: List[str] = Field(min_length=4, max_length=7)
    benefits: List[str] = Field(min_length=3, max_length=6)
    call_to_action: str = Field(min_length=15, max_length=120)


class ProductSEOResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")
    generated_at: str
    language: Literal["es-ES"]
    product: ProductDescription
    seo: SEOMetadata


## 2. Productos ficticios para las pruebas


In [10]:
PRODUCTS = [
    {
        "name": "SolarGo Backpack",
        "category": "Solar-Powered Travel Backpack",
        "price": 89.99,
        "audience": "travelers and digital nomads",
        "attributes": [
            "integrated solar panel",
            "USB charging ports",
            "water-resistant fabric",
            "anti-theft compartments",
            "30-liter capacity"
        ]
    },
    {
        "name": "PetSense Tracker",
        "category": "Smart Pet Tracker",
        "price": 69.99,
        "audience": "pet owners",
        "attributes": [
            "real-time GPS tracking",
            "activity monitoring",
            "geofencing alerts",
            "waterproof design",
            "long battery life"
        ]
    },
    {
        "name": "CleanWave Service",
        "category": "Professional Home Cleaning Service",
        "price": 79.99,
        "audience": "busy professionals and families",
        "attributes": [
            "eco-friendly products",
            "certified cleaners",
            "same-day booking",
            "flexible scheduling",
            "100% satisfaction guarantee"
        ]
    }
]
pd.DataFrame(PRODUCTS)


,name,category,price,audience,attributes
0,SolarGo Backpack,Solar-Powered Travel Backpack,89.99,travelers and digital nomads,"[integrated solar panel, USB charging ports, w..."
1,PetSense Tracker,Smart Pet Tracker,69.99,pet owners,"[real-time GPS tracking, activity monitoring, ..."
2,CleanWave Service,Professional Home Cleaning Service,79.99,busy professionals and families,"[eco-friendly products, certified cleaners, sa..."


## 3. Prompt optimizado y llamada con Structured Outputs

El prompt limita las invenciones, define criterios SEO y pide contenido en español de España. `responses.parse` aplica directamente el schema Pydantic.


In [11]:
SYSTEM_PROMPT = '''Eres especialista senior en copywriting para ecommerce y SEO en español de España.
Usa únicamente los datos facilitados. No inventes certificaciones, premios, descuentos, garantías,
compatibilidades, resultados médicos ni características técnicas. Evita keyword stuffing.
Crea un meta title de 30-60 caracteres, una meta description de 120-160, un slug válido,
entre 5 y 8 keywords con intención y prioridad, y contenido comercial natural y relevante.
Devuelve language='es-ES' y la fecha indicada en formato ISO 8601.'''


def build_prompt(product: dict) -> str:
    return f'''Genera una ficha SEO para ecommerce.
Producto: {product["name"]}
Categoría: {product["category"]}
Precio: {product["price"]:.2f} EUR
Público: {product["audience"]}
Características: {", ".join(product["attributes"])}
Fecha: {datetime.now(timezone.utc).isoformat()}
Explica características y beneficios sin añadir datos no proporcionados.'''


def generate_product_seo(product: dict) -> ProductSEOResponse:
    response = client.responses.parse(
        model=MODEL_NAME,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(product)},
        ],
        text_format=ProductSEOResponse,
        temperature=0.4,
    )
    if response.output_parsed is None:
        raise ValueError("La API no devolvió una salida estructurada procesable")
    return ProductSEOResponse.model_validate(response.output_parsed.model_dump())


## 4. Manejo de errores


In [12]:
def safe_generate(product: dict):
    try:
        result = generate_product_seo(product)
        print(f"✅ JSON válido para {product['name']}")
        return result
    except ValidationError as exc:
        print("❌ La respuesta no cumple el schema")
        print(json.dumps(exc.errors(), ensure_ascii=False, indent=2))
    except Exception as exc:
        print(f"❌ Error controlado: {type(exc).__name__}: {exc}")
        print("Configura una clave real localmente para ejecutar la llamada.")
    return None

# Con la clave ficticia esta llamada demostrará el manejo controlado del error.
result_api = safe_generate(PRODUCTS[0])


❌ Error controlado: AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-YOUR_********_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Configura una clave real localmente para ejecutar la llamada.


## 5. Validación local reproducible

Estas pruebas demuestran que el schema acepta datos correctos y rechaza valores fuera de rango, slugs incorrectos y campos adicionales.


In [14]:
VALID_EXAMPLE = {
    "generated_at": "2026-10-05T08:32:52+00:00",
    "language": "es-ES",
    "product": {
        "product_name": "SolarGo Backpack",
        "category": "Solar-Powered Travel Backpack",
        "target_audience": "Travelers and digital nomads who need connectivity and convenience while traveling.",
        "short_description": "Innovative travel backpack with an integrated solar panel and USB charging ports, designed for travelers and digital nomads.",
        "long_description": "SolarGo Backpack combines practicality, sustainability and comfort in a single travel solution. Its integrated solar panel helps keep compatible devices charged during outdoor activities, while the USB charging ports add convenience on the go. The water-resistant fabric provides additional protection in changing weather conditions, and the anti-theft compartments help secure personal belongings. With a spacious 30-liter capacity, it offers enough room for technology, clothing and travel essentials, making it a versatile option for modern travelers and digital nomads.",
        "key_features": [
            "Integrated solar panel",
            "USB charging ports",
            "Water-resistant fabric",
            "Anti-theft compartments",
            "30-liter capacity"
        ],
        "benefits": [
            "Convenient charging during trips",
            "Improved organization and security",
            "Suitable for extended travel and daily use"
        ],
        "call_to_action": "Discover SolarGo Backpack and travel smarter with sustainable charging wherever you go."
    },
    "seo": {
        "meta_title": "SolarGo Backpack | Solar Travel Backpack",
        "meta_description": "Travel backpack with integrated solar panel, USB charging ports, water-resistant fabric and anti-theft compartments for modern travelers.",
        "slug": "solargo-backpack-solar-travel-backpack",
        "keywords": [
            {
                "keyword": "solar travel backpack",
                "search_intent": "commercial",
                "priority": 5
            },
            {
                "keyword": "backpack with solar panel",
                "search_intent": "transactional",
                "priority": 5
            },
            {
                "keyword": "digital nomad backpack",
                "search_intent": "commercial",
                "priority": 4
            },
            {
                "keyword": "travel backpack with usb charging",
                "search_intent": "commercial",
                "priority": 4
            },
            {
                "keyword": "anti theft travel backpack",
                "search_intent": "commercial",
                "priority": 3
            }
        ],
        "seo_score": 92
    }
}

validated_example = ProductSEOResponse.model_validate(VALID_EXAMPLE)
print("✅ Ejemplo válido")
print(validated_example.model_dump_json(indent=2))


✅ Ejemplo válido
{
  "generated_at": "2026-10-05T08:32:52+00:00",
  "language": "es-ES",
  "product": {
    "product_name": "SolarGo Backpack",
    "category": "Solar-Powered Travel Backpack",
    "target_audience": "Travelers and digital nomads who need connectivity and convenience while traveling.",
    "short_description": "Innovative travel backpack with an integrated solar panel and USB charging ports, designed for travelers and digital nomads.",
    "long_description": "SolarGo Backpack combines practicality, sustainability and comfort in a single travel solution. Its integrated solar panel helps keep compatible devices charged during outdoor activities, while the USB charging ports add convenience on the go. The water-resistant fabric provides additional protection in changing weather conditions, and the anti-theft compartments help secure personal belongings. With a spacious 30-liter capacity, it offers enough room for technology, clothing and travel essentials, making it a ver

In [15]:
INVALID_EXAMPLE = json.loads(json.dumps(VALID_EXAMPLE))
INVALID_EXAMPLE["seo"]["seo_score"] = 140
INVALID_EXAMPLE["seo"]["slug"] = "Slug no válido"
INVALID_EXAMPLE["campo_no_permitido"] = True

try:
    ProductSEOResponse.model_validate(INVALID_EXAMPLE)
except ValidationError as exc:
    print("✅ JSON inválido rechazado correctamente")
    for error in exc.errors():
        print("-", ".".join(map(str, error["loc"])), "|", error["msg"])


✅ JSON inválido rechazado correctamente
- seo.slug | String should match pattern '^[a-z0-9]+(?:-[a-z0-9]+)*$'
- seo.seo_score | Input should be less than or equal to 100
- campo_no_permitido | Extra inputs are not permitted


## 6. Pruebas con varios productos y transformación para ecommerce


In [16]:
results = []

def to_catalog_table(items: List[ProductSEOResponse]) -> pd.DataFrame:
    return pd.DataFrame([{
        "product_name": x.product.product_name,
        "category": x.product.category,
        "short_description": x.product.short_description,
        "meta_title": x.seo.meta_title,
        "meta_description": x.seo.meta_description,
        "slug": x.seo.slug,
        "keywords": ", ".join(k.keyword for k in x.seo.keywords),
        "seo_score": x.seo.seo_score,
        "generated_at": x.generated_at,
    } for x in items])

catalog_table = to_catalog_table([validated_example, *results])
catalog_table


,product_name,category,short_description,meta_title,meta_description,slug,keywords,seo_score,generated_at
0,SolarGo Backpack,Solar-Powered Travel Backpack,Innovative travel backpack with an integrated ...,SolarGo Backpack | Solar Travel Backpack,"Travel backpack with integrated solar panel, U...",solargo-backpack-solar-travel-backpack,"solar travel backpack, backpack with solar pan...",92,2026-10-05T08:32:52+00:00
